# 06. Report figures for the models

Draws Figures 5.5 to 5.10 of the project report from `results/eval.json`, the training logs in `runs/` and the saved weights. Figures are written as PNG (300 dpi) and SVG to `docs/project_report/figures/`.

In [1]:
from pathlib import Path

# Works whether the notebook is opened from ai_model/ or ai_model/notebooks/.
ROOT = Path.cwd().resolve()
while ROOT.name != "ai_model" and ROOT != ROOT.parent:
    ROOT = ROOT.parent
assert ROOT.name == "ai_model", "Open this notebook from inside the ai_model folder"

DATA = ROOT / "data"          # downloaded and converted datasets (git-ignored)
RAW = DATA / "raw"            # original downloads
RUNS = ROOT / "runs"          # Ultralytics training runs (git-ignored)
WEIGHTS = ROOT / "weights"    # final model files used by the AI service
RESULTS = ROOT / "results"    # metrics used in the project report
for p in (DATA, RAW, RUNS, WEIGHTS, RESULTS):
    p.mkdir(parents=True, exist_ok=True)
print("ai_model root:", ROOT)

ai_model root: /Users/parth/University/Classes/3rd semester/DSN/ai_model


In [2]:
import glob, json, os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from PIL import Image, ImageDraw, ImageFont

FIG_OUT = ROOT.parent / "docs" / "project_report" / "figures"
FIG_OUT.mkdir(parents=True, exist_ok=True)

# shared print style for every report figure
INK, NAVY, BLUE, SKY, GRAY, ACCENT = "#1b1b1b", "#1f3b5c", "#3d6a99", "#a9c1da", "#7f7f7f", "#a8402f"
TEXT_W = 6.3
plt.rcParams.update({
    "font.family": "Arial", "font.size": 9, "axes.titlesize": 9.5, "axes.titleweight": "bold",
    "axes.spines.top": False, "axes.spines.right": False, "axes.linewidth": 0.7,
    "xtick.labelsize": 8, "ytick.labelsize": 8, "legend.fontsize": 8, "legend.frameon": False,
    "savefig.dpi": 300, "savefig.bbox": "tight", "savefig.pad_inches": 0.04, "svg.fonttype": "none",
})


def save(fig, name):
    for ext in ("png", "svg"):
        fig.savefig(FIG_OUT / f"{name}.{ext}", facecolor="white")
    plt.close(fig)
    print("saved", name)


def source(fig, text, y=-0.02):
    fig.text(0.0, y, text, fontsize=6.8, color=GRAY, ha="left", va="top")


def hgrid(ax):
    ax.yaxis.grid(True, color="#e3e3e3", linewidth=0.6); ax.set_axisbelow(True)


def vgrid(ax):
    ax.xaxis.grid(True, color="#e3e3e3", linewidth=0.6); ax.set_axisbelow(True)


EV = json.load(open(RESULTS / "eval.json"))
NAMES = {"pothole_road_damage": "Pothole / Road Damage", "garbage_litter": "Garbage / Litter",
         "open_manhole": "Open Manhole", "graffiti": "Graffiti"}
SHORT = {"pothole_road_damage": "Pothole", "garbage_litter": "Litter", "open_manhole": "Open manhole", "graffiti": "Graffiti"}
ORDER = [k for k in NAMES if k in EV]
VAL = {
    "pothole_road_damage": ROOT / "Pothole_Segmentation_YOLOv8.v1i.yolov8" / "valid" / "images",
    "garbage_litter": DATA / "garbage_litter" / "images" / "val",
    "open_manhole": DATA / "open_manhole" / "images" / "val",
    "graffiti": DATA / "graffiti" / "images" / "val",
}


def runcsv(k):
    df = pd.read_csv(RUNS / k / "results.csv")
    df.columns = df.columns.str.strip()
    return df

In [3]:
import torch
from ultralytics import YOLO
DEVICE = 0 if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")

In [4]:
def fig15():
    fig, axes = plt.subplots(2, 2, figsize=(TEXT_W, 4.3), gridspec_kw={"hspace": 0.55, "wspace": 0.42})
    for ax, k in zip(axes.flat, ORDER):
        df = runcsv(k)
        seg = EV[k]["task"] == "segment"
        ax.plot(df["epoch"], df["train/box_loss"], color=NAVY, lw=1.2, label="Train box loss")
        ax.plot(df["epoch"], df["val/box_loss"], color=NAVY, lw=1.2, ls="--", label="Val box loss")
        if seg:
            ax.plot(df["epoch"], df["train/seg_loss"], color=SKY, lw=1.2, label="Train mask loss")
            ax.plot(df["epoch"], df["val/seg_loss"], color=SKY, lw=1.2, ls="--", label="Val mask loss")
        ax.set_xlabel("Epoch")
        ax.set_ylabel("Loss")
        hgrid(ax)
        a2 = ax.twinx()
        col = "metrics/mAP50(M)" if seg else "metrics/mAP50(B)"
        a2.plot(df["epoch"], df[col], color=ACCENT, lw=1.4)
        a2.set_ylim(0, 1)
        a2.set_ylabel("mAP50 (" + ("mask" if seg else "box") + ")", color=ACCENT)
        a2.tick_params(axis="y", colors=ACCENT)
        a2.spines["right"].set_visible(True)
        a2.spines["right"].set_color(ACCENT)
        best = EV[k]["best_epoch"]
        a2.axvline(best, color=GRAY, lw=0.6, ls=":")
        ax.set_title(f"{NAMES[k]} ({'segmentation' if seg else 'detection'})", loc="left", fontsize=8.6)
    h = [Line2D([], [], color=NAVY, lw=1.2, label="Box loss"), Line2D([], [], color=SKY, lw=1.2, label="Mask loss"),
         Line2D([], [], color=INK, lw=1.2, label="Train"), Line2D([], [], color=INK, lw=1.2, ls="--", label="Validation"),
         Line2D([], [], color=ACCENT, lw=1.4, label="Validation mAP50"), Line2D([], [], color=GRAY, lw=0.6, ls=":", label="Best epoch")]
    fig.legend(handles=h, loc="lower center", ncol=6, bbox_to_anchor=(0.5, -0.03), handlelength=1.6, columnspacing=1.0)
    save(fig, "fig15_training_curves")

In [5]:
def fig16():
    fig, axes = plt.subplots(1, len(ORDER), figsize=(TEXT_W, 1.95), sharey=True, gridspec_kw={"wspace": 0.12})
    for ax, k in zip(np.atleast_1d(axes), ORDER):
        cv = EV[k]["curves"]
        for key, c in cv.items():
            mask = "(M)" in key
            ap = EV[k]["mask" if mask else "box"]["map50"]
            ax.plot(c["x"], c["y"], color=SKY if mask else NAVY, lw=1.4, label=f"{'Mask' if mask else 'Box'} AP50 {ap:.2f}")
            ax.fill_between(c["x"], c["y"], color=SKY if mask else NAVY, alpha=0.07)
        ax.set_xlim(0, 1); ax.set_ylim(0, 1.02)
        ax.set_xticks([0, 0.5, 1], ["0", "0.5", "1"])
        ax.set_xlabel("Recall")
        ax.set_title(SHORT[k], fontsize=8, loc="left")
        ax.legend(loc="lower left", fontsize=6.4, handlelength=1.1, frameon=True, framealpha=0.9, edgecolor="none")
        hgrid(ax)
    np.atleast_1d(axes)[0].set_ylabel("Precision")
    save(fig, "fig16_precision_recall")

In [6]:
def fig17():
    fig, axes = plt.subplots(1, len(ORDER), figsize=(TEXT_W, 1.95), gridspec_kw={"wspace": 0.6})
    for ax, k in zip(np.atleast_1d(axes), ORDER):
        cm = np.array(EV[k]["confusion"], dtype=float)  # rows predicted, cols true: [[TP, FP], [FN, -]]
        tp, fp, fn = cm[0, 0], cm[0, 1], cm[1, 0]
        rec = tp / max(tp + fn, 1)
        shade = np.array([[rec, 0.12], [1 - rec, 0.0]])
        ax.imshow(shade, cmap="Blues", vmin=0, vmax=1)
        txt = [[f"TP {int(tp)}\n{rec:.2f}", f"FP {int(fp)}"], [f"FN {int(fn)}\n{1 - rec:.2f}", "n/a"]]
        for i in range(2):
            for j in range(2):
                ax.text(j, i, txt[i][j], ha="center", va="center", fontsize=6.8,
                        color="white" if shade[i, j] > 0.55 else INK, linespacing=1.15)
        ax.set_xticks([0, 1], ["defect", "background"], fontsize=6.8)
        ax.set_yticks([0, 1], ["defect", "background"], fontsize=6.8, rotation=90, va="center")
        ax.set_xlabel("True", fontsize=7.4)
        ax.set_ylabel("Predicted", fontsize=7.4)
        prec = tp / max(tp + fp, 1)
        ax.set_title(f"{NAMES[k]}\nP {prec:.2f}, R {rec:.2f}", fontsize=7.4, loc="left", linespacing=1.2)
        for sp in ax.spines.values():
            sp.set_visible(False)
        ax.tick_params(length=0)
    source(fig, "Instance counts at confidence 0.25, IoU 0.45. The defect column is normalised by true defects. "
                "Detectors have no background-background cell.", y=-0.02)
    save(fig, "fig17_confusion_matrices")

In [7]:
def fig18():
    metrics = [("p", "Precision"), ("r", "Recall"), ("map50", "mAP50"), ("map5095", "mAP50-95")]
    fig, ax = plt.subplots(figsize=(TEXT_W, 2.5))
    x = np.arange(len(ORDER))
    w = 0.19
    cols = [SKY, BLUE, NAVY, "#9aa3ad"]
    for i, (mk, ml) in enumerate(metrics):
        vals = [EV[k]["mask" if EV[k]["task"] == "segment" else "box"][mk] for k in ORDER]
        b = ax.bar(x + (i - 1.5) * w, vals, w, color=cols[i], label=ml)
        for bb, v in zip(b, vals):
            ax.text(bb.get_x() + bb.get_width() / 2, v + 0.012, f"{v:.2f}", ha="center", fontsize=6.4)
    ax.set_xticks(x, [NAMES[k] + ("\n(mask)" if EV[k]["task"] == "segment" else "\n(box)") for k in ORDER])
    ax.set_ylim(0, 1.05)
    ax.set_ylabel("Score")
    hgrid(ax)
    ax.legend(loc="upper center", ncol=4, bbox_to_anchor=(0.5, 1.14))
    save(fig, "fig18_model_comparison")

In [8]:
def fig19():
    fig, ax = plt.subplots(figsize=(TEXT_W, 2.1))
    y = np.arange(len(ORDER))[::-1]
    h = 0.34
    for yi, k in zip(y, ORDER):
        c, g = EV[k]["latency"]["cpu"]["mean_ms"], EV[k]["latency"]["gpu"]["mean_ms"]
        ax.barh(yi + h / 2, c, h, color=SKY)
        ax.barh(yi - h / 2, g, h, color=NAVY)
        ax.text(c + 0.5, yi + h / 2, f"{c:.0f} ms", va="center", fontsize=7.2)
        ax.text(g + 0.5, yi - h / 2, f"{g:.0f} ms", va="center", fontsize=7.2)
    ax.set_yticks(y, [NAMES[k] for k in ORDER])
    ax.set_xlabel("Mean time per 640 px image, including pre- and post-processing (ms)")
    mx = max(EV[k]["latency"]["cpu"]["mean_ms"] for k in ORDER)
    ax.set_xlim(0, mx * 1.25)
    vgrid(ax)
    ax.tick_params(axis="y", length=0)
    ax.spines["left"].set_visible(False)
    ax.legend(handles=[Patch(color=SKY, label="CPU (Apple M5)"), Patch(color=NAVY, label="GPU (Apple MPS)")], loc="lower right")
    source(fig, "Mean over 40 validation images after 3 warm-up runs, batch size 1.", y=-0.13)
    save(fig, "fig19_inference_latency")

In [9]:
def fig20():
    tiles = []
    fnt = ImageFont.truetype("/System/Library/Fonts/Supplemental/Arial Bold.ttf", 30)
    for k in ORDER:
        model = YOLO(str(WEIGHTS / f"{k}_model.pt"))
        files = [f for f in sorted(glob.glob(str(VAL[k] / "*.jpg"))) if "aug" not in os.path.basename(f).lower()]
        picked = []
        for f in files:
            r = model.predict(f, imgsz=640, conf=0.35, device=DEVICE, verbose=False)[0]
            if r.boxes is not None and 1 <= len(r.boxes) <= 4 and float(r.boxes.conf.max()) > 0.6:
                arr = np.asarray(Image.open(f).convert("L"))
                if (arr.mean(1) < 12).mean() > 0.05 or (arr.mean(0) < 12).mean() > 0.05:
                    continue
                picked.append((f, r))
            if len(picked) == 2:
                break
        for f, r in picked:
            im0 = Image.open(f).convert("RGB")
            sc = 1000 / max(im0.size)
            im = im0.resize((round(im0.size[0] * sc), round(im0.size[1] * sc)), Image.LANCZOS)
            W, H = im.size
            over = Image.new("RGBA", im.size, (0, 0, 0, 0))
            d = ImageDraw.Draw(over)
            if r.masks is not None:
                for poly in r.masks.xy:
                    if len(poly) > 2:
                        d.polygon([tuple(p * sc) for p in poly], fill=(168, 64, 47, 90), outline=(168, 64, 47, 255))
            for (x0, y0, x1, y1), c in zip((np.array(r.boxes.xyxy.tolist()) * sc).tolist(), r.boxes.conf.tolist()):
                lw = 5
                d.rectangle([x0, y0, x1, y1], outline=(255, 214, 0, 255), width=lw)
                t = f"{c:.2f}"
                tw = d.textlength(t, font=fnt)
                d.rectangle([x0, max(0, y0 - 38), x0 + tw + 14, max(38, y0)], fill=(255, 214, 0, 255))
                d.text((x0 + 7, max(0, y0 - 38) + 3), t, fill=(0, 0, 0, 255), font=fnt)
            im = Image.alpha_composite(im.convert("RGBA"), over).convert("RGB")
            bx = np.array(r.boxes.xyxy.tolist()) * sc
            cx, cy = (bx[:, 0].min() + bx[:, 2].max()) / 2, (bx[:, 1].min() + bx[:, 3].max()) / 2
            cw = min(W, H * 4 / 3); ch = cw * 3 / 4
            x0 = min(max(cx - cw / 2, 0), W - cw); y0 = min(max(cy - ch / 2, 0), H - ch)
            im = im.crop((int(x0), int(y0), int(x0 + cw), int(y0 + ch))).resize((480, 360), Image.LANCZOS)
            tiles.append((k, im))
    n = len(tiles)
    cols = 4
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(TEXT_W, 1.32 * rows + 0.1), gridspec_kw={"wspace": 0.04, "hspace": 0.2})
    fig.subplots_adjust(left=0.005, right=0.995)
    for ax in np.atleast_1d(axes).flat:
        ax.axis("off")
    for ax, (k, t) in zip(np.atleast_1d(axes).flat, tiles):
        ax.imshow(t)
        ax.set_title(SHORT[k], fontsize=7.6)
    source(fig, "Yellow boxes: detections with confidence. Red fill: predicted masks (segmentation models). Images from each validation set.", y=0.0)
    save(fig, "fig20_sample_detections")

In [10]:
fig15(); fig16(); fig17(); fig18(); fig19(); fig20()

saved fig15_training_curves
saved fig16_precision_recall


saved fig17_confusion_matrices
saved fig18_model_comparison
saved fig19_inference_latency


saved fig20_sample_detections
